## Project 1

### Part A

In [ ]:
import pandas as pd
# Code part A
# Taking the 10 year averaged datasets for offshore wind, onshore wind and pv

offshore_wind_cf = pd.read_csv("../data/averaged_offshore_wind_capacity_factor_denmark_hourly.csv", index_col=0)
onshore_wind_cf = pd.read_csv("../data/averaged_onshore_wind_capacity_factor_denmark_hourly.csv", index_col=0)
pv_cf = pd.read_csv("../data/averaged_pv_capacity_factor_denmark_hourly.csv", index_col=0)
demand = pd.read_csv("../data/denmark_demand.csv", index_col=0)

pv = pv_cf.rename(columns={"cf": "pv_cf"})
onshore = onshore_wind_cf.rename(columns={"cf": "onshore_cf"})
offshore = offshore_wind_cf.rename(columns={"cf": "offshore_cf"})

work_df = pd.concat([pv, onshore, offshore, demand], axis=1)

In [ ]:
# Annualised capital costs
annualised_offshore_wind_capex = 101644 # EUR/MW/year PLACEHOLDER
annualised_onshore_wind_capex = 101644 # EUR/MW/year PLACEHOLDER
annualised_pv_capex = 51346 # EUR/MW/year PLACEHOLDER
annualised_battery_capex = 0 # EUR/MW/year PLACEHOLDER

# Annual OPEX costs
offshore_wind_opex = 0 # EUR/MW/year PLACEHOLDER
onshore_wind_opex = 0 # EUR/MW/year PLACEHOLDER
pv_opex = 0 # EUR/MW/year PLACEHOLDER
battery_opex = 0 # EUR/MW/year PLACEHOLDER

# Setting up marginal generation costs
onshore_wind_marginal_cost = 0 # EUR/MWh PLACEHOLDER
offshore_wind_marginal_cost = 0 # EUR/MWh PLACEHOLDER
pv_marginal_cost = 0 # EUR/MWh PLACEHOLDER
battery_marginal_cost = 0 # EUR/MW/year PLACEHOLDER

In [ ]:
# Building the model

import linopy
import matplotlib.pyplot as plt

# Create model instance
m = linopy.Model()

# Take time index to use as a time coordiante (this is a timeseries)
T = work_df.index

# Take capacity factors of technologies
cf_pv = work_df["pv_cf"]
cf_off_wind = work_df["offshore_cf"]
cf_on_wind = work_df["onshore_cf"]
demand = work_df["demand"]

# Add capacity variable (MW)
K_pv   = m.add_variables(lower=0, name="K_pv")
K_offshore_wind = m.add_variables(lower=0, name="K_offshore_wind")
K_onshore_wind = m.add_variables(lower=0, name="K_onshore_wind")

# Add dispatch variables (MW) over time
g_pv   = m.add_variables(lower=0, coords=[T], name="g_pv")
g_offshore_wind = m.add_variables(lower=0, coords=[T], name="g_offshore_wind")
g_onshore_wind = m.add_variables(lower=0, coords=[T], name="g_onshore_wind")

# Add constraint that g_pv (dispatch at given time) can't be more than capacity * capacity factor
m.add_constraints(g_pv <= K_pv * cf_pv, name="pv_cf_limit")
# Add constraint that g_offshore_wind (dispatch at given time) can't be more than capacity * capacity factor
m.add_constraints(g_offshore_wind <= K_offshore_wind * cf_off_wind, name="offshore wind_cf_limit")
# Add constraint that g_onshore_wind (dispatch at given time) can't be more than capacity * capacity factor
m.add_constraints(g_onshore_wind <= K_onshore_wind * cf_on_wind, name="onshore wind_cf_limit")

# Add constraint that production must meet demand balance each hour
m.add_constraints(g_pv + g_offshore_wind + g_onshore_wind == demand, name="power_balance")

# Create objective function
capex = (
    annualised_pv_capex   * K_pv +
    annualised_offshore_wind_capex * K_offshore_wind +
    annualised_onshore_wind_capex * K_onshore_wind
)

opex = (
    pv_marginal_cost * (g_pv.sum()) +
    onshore_wind_marginal_cost * (g_onshore_wind.sum()) +
    offshore_wind_marginal_cost * (g_offshore_wind.sum())
)

m.add_objective(capex + opex, sense="min")
solver = "highs" # Choose between: gurobi,highs
result = m.solve(solver_name=solver)

In [8]:
print("Solver status:", result)
print("Objective value:", float(m.objective.value))

# Extract optimal capacities
K_pv_sol = float(m.variables["K_pv"].solution)
K_offshore_wind_sol = float(m.variables["K_offshore_wind"].solution)
K_onshore_wind_sol = float(m.variables["K_onshore_wind"].solution)

print("Optimal installed capacities (MW):")
print(f"PV   : {K_pv_sol:.3f}")
print(f"Offshore Wind : {K_offshore_wind_sol:.3f}")
print(f"Onshore Wind : {K_onshore_wind_sol:.3f}")

Solver status: ('ok', 'optimal')
Objective value: 4033330214.315793
Optimal installed capacities (MW):
PV   : 0.000
Offshore Wind : 39680.947
Onshore Wind : 0.000


In [ ]:
# Plots part A
# Martin


Solver status: ('ok', 'optimal')
Objective value: 4033330214.315793


### Part B

In [ ]:
import pypsa
import pandas as pd
import numpy as np

# ----------------------------
# Create time snapshots
# ----------------------------
hours = pd.date_range("2025-01-01 00:00", periods=24, freq="h")

# Demand profile
load = pd.Series(100.0, index=hours)

# ----------------------------
# Weather year A
# ----------------------------
wind_cf_A = pd.Series(
[0.45,0.42,0.40,0.38,0.35,0.33,0.30,0.32,0.36,0.40,0.43,0.46,
0.48,0.50,0.47,0.44,0.41,0.39,0.37,0.36,0.38,0.41,0.44,0.46],
index=hours)

solar_cf_A = pd.Series(
[0,0,0,0,0,0.02,0.10,0.25,0.45,0.60,0.70,0.75,
0.72,0.65,0.50,0.30,0.12,0.02,0,0,0,0,0,0],
index=hours)

# ----------------------------
# Weather year B
# ----------------------------
wind_cf_B = pd.Series(
[0.22,0.20,0.18,0.17,0.16,0.15,0.14,0.16,0.18,0.20,0.22,0.24,
0.26,0.28,0.27,0.25,0.23,0.22,0.21,0.20,0.21,0.22,0.23,0.24],
index=hours)

solar_cf_B = pd.Series(
[0,0,0,0,0,0.05,0.18,0.35,0.55,0.72,0.82,0.88,
0.84,0.76,0.60,0.40,0.20,0.06,0,0,0,0,0,0],
index=hours)

# ----------------------------
# Model function
# ----------------------------
def run_model(wind_cf, solar_cf, label):

    n = pypsa.Network()
    n.set_snapshots(hours)

    n.add("Bus", "electricity", carrier="electricity")

    n.add("Load",
          "demand",
          bus="electricity",
          p_set=load)

    n.add("Generator",
          "wind",
          bus="electricity",
          p_max_pu=wind_cf,
          capital_cost=600,
          marginal_cost=0,
          p_nom_extendable=True)

    n.add("Generator",
          "solar",
          bus="electricity",
          p_max_pu=solar_cf,
          capital_cost=400,
          marginal_cost=0,
          p_nom_extendable=True)

    n.add("Generator",
          "gas",
          bus="electricity",
          p_max_pu=1,
          capital_cost=800,
          marginal_cost=70,
          p_nom_extendable=True)

    n.optimize()

    result = {
        "year": label,
        "avg_wind_cf": wind_cf.mean(),
        "avg_solar_cf": solar_cf.mean(),
        "wind_capacity": n.generators.at["wind","p_nom_opt"],
        "solar_capacity": n.generators.at["solar","p_nom_opt"],
        "gas_capacity": n.generators.at["gas","p_nom_opt"]
    }

    return n, result


# ----------------------------
# Run both weather years
# ----------------------------
nA, resA = run_model(wind_cf_A, solar_cf_A, "Year A")
nB, resB = run_model(wind_cf_B, solar_cf_B, "Year B")

results = pd.DataFrame([resA,resB])

print(results)

In [ ]:
import matplotlib.pyplot as plt

# ----------------------------
# Capacity factors
# ----------------------------
cf_table = results[["year","avg_wind_cf","avg_solar_cf"]].set_index("year")

cf_table.plot(kind="bar")
plt.title("Average renewable capacity factors by year")
plt.ylabel("Capacity factor")
plt.show()


# ----------------------------
# Installed capacities
# ----------------------------
capacity_table = results[
["year","wind_capacity","solar_capacity","gas_capacity"]
].set_index("year")

capacity_table.plot(kind="bar")
plt.title("Optimized installed capacities")
plt.ylabel("MW")
plt.show()


# ----------------------------
# Dispatch plots
# ----------------------------
dispatch_A = nA.generators_t.p[["wind","solar","gas"]]
dispatch_A.plot()
plt.title("Dispatch Year A")
plt.ylabel("MW")
plt.show()

dispatch_B = nB.generators_t.p[["wind","solar","gas"]]
dispatch_B.plot()
plt.title("Dispatch Year B")
plt.ylabel("MW")
plt.show()

### Part C

In [ ]:
# Code part C

In [ ]:
# Plots part C

### Part D

In [ ]:
# Code part D

In [ ]:
# Plots part D

### Part E